In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

##################################
# GARCH-X，照計畫書的公式，但效果很差， VAR的 STD 是用 GARCH的SHAPE

In [ ]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"


In [ ]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel

In [ ]:
# =============================================================================
# ES1 GK-LSTM VaR-t  （Vol 版）
# Target Y : gk_vol_daily  (Garman-Klass 單日波動度，已是 σ 尺度，無需再開根號)
# Features  : ES1_LN_RET, gk_vol_daily(lag), garch_vol, VIX_CLOSE
# Split     : Train 2006-2021 / Test 2022-2025
# VaR       : t 分配, mu=0, shape clip(6,10)
# Backtest  : Kupiec UC + Christoffersen CC (訓練集 & 測試集)
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import t as tdist, chi2

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# ── 可選：GPU 設定（有 GPU 就自動加速；沒有 GPU 也可正常跑）─────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPU found: {len(gpus)} device(s)")
else:
    print("ℹ No GPU found, running on CPU")


# =============================================================================
# 0. 設定區  ── 修改這裡即可調整所有超參數
# =============================================================================
# 資料路徑
PATH_DAY  = "./filtered_output/df_day.csv"                   # df_day.csv (ES1 + VIX)
PATH_VOL  = "./real_volatility_multi_var/es1_volatility_all_methods.csv"  # GK / GARCH / shape

# 輸出目錄
OUTPUT_DIR = "./LSTM_diagnostics"

# 樣本切分
TRAIN_END  = "2021-12-31"   # 訓練集最後一天
TEST_START = "2022-01-01"   # 測試集起點

# LSTM 超參數
LOOKBACK   = 20          # 滑動視窗天數
LSTM_UNITS = 64          # LSTM 隱藏層神經元數
DROPOUT    = 0.2         # Dropout 比率
LR         = 1e-3        # Adam 學習率
EPOCHS     = 100         # 最大 epoch（配合 EarlyStopping）
BATCH_SIZE = 32
PATIENCE   = 20          # EarlyStopping patience

# VaR 設定
ALPHA_95   = 0.05
ALPHA_99   = 0.01
NU_CLIP    = (6, 10)     # shape 截尾範圍

# 特徵欄位（順序固定，對應 build_sequences 內的 X）
FEATURE_COLS = ['ES1_LN_RET', 'gk_vol_daily', 'garch_vol', 'VIX_CLOSE']
TARGET_COL   = 'gk_vol_daily'

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 1. 讀資料 & 建母資料表
# =============================================================================
def load_master(path_day: str, path_vol: str) -> pd.DataFrame:
    # """
    # 合併 df_day.csv 與 es1_volatility_all_methods.csv，
    # 以 DATE 對齊，篩選 2006 年之後。
    # """
    # --- df_day ---
    day = pd.read_csv(path_day, parse_dates=['DATE'])
    # 若 DATE 是 index（讀進來可能有 index 欄）
    if 'DATE' not in day.columns:
        day = day.reset_index().rename(columns={'index': 'DATE'})
    day['DATE'] = pd.to_datetime(day['DATE'], errors='coerce').dt.normalize()

    keep_day = ['DATE', 'ES1_LN_RET', 'ES1_CLOSE', 'ES1_VOLUME', 'VIX_LN_RET', 'VIX_CLOSE']
    day = day[[c for c in keep_day if c in day.columns]].dropna(subset=['DATE'])

    # --- vol / garch ---
    vol = pd.read_csv(path_vol, parse_dates=['DATE'])
    vol['DATE'] = pd.to_datetime(vol['DATE'], errors='coerce').dt.normalize()
    keep_vol = ['DATE', 'gk_vol_daily', 'garch_vol', 'shape']
    vol = vol[[c for c in keep_vol if c in vol.columns]].dropna(subset=['DATE'])

    # --- merge ---
    df = pd.merge(day, vol, on='DATE', how='inner')
    df = df.sort_values('DATE').reset_index(drop=True)

    # 篩選 2006 以後
    df = df[df['DATE'] >= '2006-01-01'].reset_index(drop=True)

    # 刪除任何特徵缺值
    df = df.dropna(subset=FEATURE_COLS + ['shape', 'ES1_LN_RET']).reset_index(drop=True)

    print(f"Master table: {df.shape[0]} rows  "
          f"({df['DATE'].min().date()} ~ {df['DATE'].max().date()})")
    return df


df = load_master(PATH_DAY, PATH_VOL)


# =============================================================================
# 2. 特徵縮放 & 滑動視窗序列建構
# =============================================================================
# 以訓練集計算 scaler（避免 data leakage）
train_mask = df['DATE'] < TEST_START

scaler_X = MinMaxScaler()
scaler_X.fit(df.loc[train_mask, FEATURE_COLS])
X_scaled_all = scaler_X.transform(df[FEATURE_COLS].values)

scaler_y = MinMaxScaler()
scaler_y.fit(df.loc[train_mask, [TARGET_COL]])
Y_scaled_all = scaler_y.transform(df[[TARGET_COL]].values).ravel()


def build_sequences(X: np.ndarray, Y: np.ndarray,
                    dates: np.ndarray, shapes: np.ndarray,
                    returns: np.ndarray, true_vol: np.ndarray,
                    lookback: int):
    # """
    # 滑動視窗：
    #   X[i] = X[i-lookback : i]   (過去 lookback 天的特徵)
    #   Y[i] = Y[i]                (第 i 天的 gk_vol_daily，已縮放)
    # """
    Xs, Ys = [], []
    seq_dates, seq_shapes, seq_ret, seq_vol = [], [], [], []

    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback: i])   # shape: (lookback, n_features)
        Ys.append(Y[i])
        seq_dates.append(dates[i])
        seq_shapes.append(shapes[i])
        seq_ret.append(returns[i])
        seq_vol.append(true_vol[i])

    return (np.array(Xs, dtype=np.float32),
            np.array(Ys, dtype=np.float32),
            np.array(seq_dates),
            np.array(seq_shapes, dtype=np.float32),
            np.array(seq_ret,   dtype=np.float32),
            np.array(seq_vol,   dtype=np.float32))


X_seq, Y_seq, seq_dates, seq_shapes, seq_ret, seq_vol = build_sequences(
    X_scaled_all,
    Y_scaled_all,
    df['DATE'].values,
    df['shape'].values,
    df['ES1_LN_RET'].values,
    df[TARGET_COL].values,
    LOOKBACK
)

# 訓練 / 測試切分（以目標日期決定）
tr_mask = pd.to_datetime(seq_dates) < pd.Timestamp(TEST_START)
te_mask = ~tr_mask

X_train, Y_train = X_seq[tr_mask], Y_seq[tr_mask]
X_test,  Y_test  = X_seq[te_mask], Y_seq[te_mask]
dates_tr = pd.to_datetime(seq_dates[tr_mask])
dates_te = pd.to_datetime(seq_dates[te_mask])
shape_tr, shape_te   = seq_shapes[tr_mask], seq_shapes[te_mask]
ret_tr,   ret_te     = seq_ret[tr_mask],    seq_ret[te_mask]
true_vol_tr, true_vol_te = seq_vol[tr_mask], seq_vol[te_mask]

print(f"Train: {X_train.shape}  ({dates_tr[0].date()} ~ {dates_tr[-1].date()})")
print(f"Test : {X_test.shape}   ({dates_te[0].date()} ~ {dates_te[-1].date()})")


# =============================================================================
# 3. 建立 LSTM 模型  (tensorflow.keras)
# =============================================================================
def build_lstm(lookback: int, n_features: int) -> tf.keras.Model:
    # """
    # 單層 LSTM + Dropout + Dense(softplus)
    # softplus 保證輸出恆正，與 volatility 尺度一致。
    # """
    inp = layers.Input(shape=(lookback, n_features), name="input_seq")
    x   = layers.LSTM(LSTM_UNITS, return_sequences=False, name="lstm")(inp)
    x   = layers.Dropout(DROPOUT, name="dropout")(x)
    out = layers.Dense(1, activation='softplus', name="output")(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LR),
        loss='huber'       # Huber loss：對極端波動更穩健（等價 delta=1）
    )
    model.summary()
    return model


n_features = X_train.shape[2]
model = build_lstm(LOOKBACK, n_features)


# =============================================================================
# 4. 訓練
# =============================================================================
cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=10, min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_train, Y_train,
    validation_split=0.1,        # 訓練集末 10% 為驗證集（時序保留）
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,               # 時間序列不打亂
    callbacks=cb_list,
    verbose=1
)

print(f"\n✓ Training finished at epoch {len(history.history['loss'])}")


# =============================================================================
# 5. 波動預測 & 反縮放
# =============================================================================
def descale_vol(y_scaled: np.ndarray) -> np.ndarray:
    # """將縮放後的預測值還原至原始 vol 尺度，並 clip 到正值。"""
    y_orig = scaler_y.inverse_transform(y_scaled.reshape(-1, 1)).ravel()
    return np.clip(y_orig, 1e-8, None)


pred_tr_scaled = model.predict(X_train, verbose=0).ravel()
pred_te_scaled = model.predict(X_test,  verbose=0).ravel()

pred_tr = descale_vol(pred_tr_scaled)   # 訓練集預測 vol
pred_te = descale_vol(pred_te_scaled)   # 測試集預測 vol


# =============================================================================
# 6. 波動預測評估指標（RMSE / MAE / Corr）
# =============================================================================
def vol_metrics(true: np.ndarray, pred: np.ndarray, label: str) -> dict:
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    mae  = float(mean_absolute_error(true, pred))
    corr = float(np.corrcoef(true, pred)[0, 1])
    print(f"[{label}]  RMSE={rmse:.6f}  MAE={mae:.6f}  Corr={corr:.4f}")
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'Corr': corr}


metrics_tr = vol_metrics(true_vol_tr, pred_tr, "Train 2006-2021")
metrics_te = vol_metrics(true_vol_te, pred_te, "Test  2022-2025")

df_vol_metrics = pd.DataFrame([metrics_tr, metrics_te])
df_vol_metrics.to_csv(os.path.join(OUTPUT_DIR, "vol_forecast_metrics.csv"),
                      index=False, encoding='utf-8-sig')
print("✓ Saved vol_forecast_metrics.csv")


# =============================================================================
# 7. VaR-t 建構
# =============================================================================
def build_var_t(sigma: np.ndarray, shape: np.ndarray, alpha: float) -> np.ndarray:
    # """
    # VaR = sigma × s × t_ν^{-1}(alpha)
    # s = sqrt((ν-2)/ν)   [Var=1 標準化]
    # ν = clip(shape, NU_CLIP)
    # """
    nu = np.clip(shape, *NU_CLIP)
    s  = np.sqrt((nu - 2.0) / nu)
    q  = tdist.ppf(alpha, df=nu)      # 負值
    return sigma * s * q              # 負值（左尾損失）


var95_tr = build_var_t(pred_tr, shape_tr, ALPHA_95)
var99_tr = build_var_t(pred_tr, shape_tr, ALPHA_99)

var95_te = build_var_t(pred_te, shape_te, ALPHA_95)
var99_te = build_var_t(pred_te, shape_te, ALPHA_99)

# 違規 (1 = 實際報酬 < VaR)
viol95_tr = (ret_tr < var95_tr).astype(int)
viol99_tr = (ret_tr < var99_tr).astype(int)
viol95_te = (ret_te < var95_te).astype(int)
viol99_te = (ret_te < var99_te).astype(int)


# =============================================================================
# 8. 回測統計函數（Kupiec + Christoffersen）── 與你原本的版本相同
# =============================================================================
def kupiec_test(viol: np.ndarray, alpha: float) -> dict:
    # """Kupiec POF / Unconditional Coverage Test"""
    v = np.asarray(viol, dtype=int)
    n = len(v);  x = int(v.sum())
    eps = 1e-12
    p_hat = np.clip(x / n, eps, 1 - eps)
    a_c   = np.clip(alpha, eps, 1 - eps)

    ll_h0  = (n - x) * np.log(1 - a_c)  + x * np.log(a_c)
    ll_mle = (n - x) * np.log(1 - p_hat) + x * np.log(p_hat)
    LR_uc  = -2.0 * (ll_h0 - ll_mle)
    p_val  = 1.0 - chi2.cdf(LR_uc, df=1)

    return dict(alpha=alpha, n=n, x=x, viol_rate=float(x/n),
                LR_uc=float(LR_uc), p_value=float(p_val))


def christoffersen_cc_test(viol: np.ndarray, alpha: float) -> dict:
    # """Christoffersen Conditional Coverage (UC + IND) Test"""
    v = np.asarray(viol, dtype=int)
    n = len(v)
    uc = kupiec_test(v, alpha)
    LR_uc = uc['LR_uc']

    v_lag, v_now = v[:-1], v[1:]
    n00 = int(np.sum((v_lag==0) & (v_now==0)))
    n01 = int(np.sum((v_lag==0) & (v_now==1)))
    n10 = int(np.sum((v_lag==1) & (v_now==0)))
    n11 = int(np.sum((v_lag==1) & (v_now==1)))

    eps   = 1e-12
    pi01  = np.clip(n01 / max(n00+n01, 1), eps, 1-eps)
    pi11  = np.clip(n11 / max(n10+n11, 1), eps, 1-eps)
    pi    = np.clip((n01+n11) / max(n00+n01+n10+n11, 1), eps, 1-eps)

    ll_h0 = (n00+n10)*np.log(1-pi)  + (n01+n11)*np.log(pi)
    ll_h1 = n00*np.log(1-pi01) + n01*np.log(pi01) \
          + n10*np.log(1-pi11) + n11*np.log(pi11)

    LR_ind = -2.0 * (ll_h0 - ll_h1)
    p_ind  = 1.0 - chi2.cdf(LR_ind, df=1)
    LR_cc  = LR_uc + LR_ind
    p_cc   = 1.0 - chi2.cdf(LR_cc, df=2)

    return dict(
        alpha=alpha, n=n, x=uc['x'], viol_rate=uc['viol_rate'],
        n00=n00, n01=n01, n10=n10, n11=n11,
        LR_uc=LR_uc,  p_uc=uc['p_value'],
        LR_ind=float(LR_ind), p_ind=float(p_ind),
        LR_cc=float(LR_cc),   p_cc=float(p_cc),
    )


# =============================================================================
# 9. 跑回測 & 整理輸出表
# =============================================================================
def run_backtest(viol95, viol99, label):
    k95 = kupiec_test(viol95, ALPHA_95)
    c95 = christoffersen_cc_test(viol95, ALPHA_95)
    k99 = kupiec_test(viol99, ALPHA_99)
    c99 = christoffersen_cc_test(viol99, ALPHA_99)

    print(f"\n{'='*55}")
    print(f"[{label}] VaR95 — N={k95['n']}, Violations={k95['x']}, "
          f"Rate={k95['viol_rate']*100:.2f}%")
    print(f"  Kupiec UC : LR={k95['LR_uc']:.4f}, p={k95['p_value']:.4f}  "
          f"{'✓ PASS' if k95['p_value']>=0.05 else '✗ FAIL'}")
    print(f"  CC test   : LR={c95['LR_cc']:.4f}, p={c95['p_cc']:.4f}  "
          f"{'✓ PASS' if c95['p_cc']>=0.05 else '✗ FAIL'}")
    print(f"  IND test  : p={c95['p_ind']:.4f}")
    print(f"  Transitions n00={c95['n00']} n01={c95['n01']} "
          f"n10={c95['n10']} n11={c95['n11']}")

    print(f"\n[{label}] VaR99 — N={k99['n']}, Violations={k99['x']}, "
          f"Rate={k99['viol_rate']*100:.2f}%")
    print(f"  Kupiec UC : LR={k99['LR_uc']:.4f}, p={k99['p_value']:.4f}  "
          f"{'✓ PASS' if k99['p_value']>=0.05 else '✗ FAIL'}")
    print(f"  CC test   : LR={c99['LR_cc']:.4f}, p={c99['p_cc']:.4f}  "
          f"{'✓ PASS' if c99['p_cc']>=0.05 else '✗ FAIL'}")
    print(f"  Transitions n00={c99['n00']} n01={c99['n01']} "
          f"n10={c99['n10']} n11={c99['n11']}")

    return (k95, c95, k99, c99)


k95_tr, c95_tr, k99_tr, c99_tr = run_backtest(viol95_tr, viol99_tr, "TRAIN 2006-2021")
k95_te, c95_te, k99_te, c99_te = run_backtest(viol95_te, viol99_te, "TEST  2022-2025")


def make_summary_row(k_dict, c_dict, split, level):
    return {
        'split': split, 'level': level,
        'N': k_dict['n'], 'violations': k_dict['x'],
        'viol_rate': k_dict['viol_rate'],
        'kupiec_LR': k_dict['LR_uc'], 'kupiec_p': k_dict['p_value'],
        'cc_LR': c_dict['LR_cc'],     'cc_p': c_dict['p_cc'],
        'ind_p': c_dict['p_ind'],
        'n00': c_dict['n00'], 'n01': c_dict['n01'],
        'n10': c_dict['n10'], 'n11': c_dict['n11'],
        'uc_pass': k_dict['p_value'] >= 0.05,
        'cc_pass': c_dict['p_cc'] >= 0.05,
    }


df_summary = pd.DataFrame([
    make_summary_row(k95_tr, c95_tr, 'train', '95'),
    make_summary_row(k99_tr, c99_tr, 'train', '99'),
    make_summary_row(k95_te, c95_te, 'test',  '95'),
    make_summary_row(k99_te, c99_te, 'test',  '99'),
])
df_summary.to_csv(os.path.join(OUTPUT_DIR, "backtest_summary.csv"),
                  index=False, encoding='utf-8-sig')
print("\n✓ Saved backtest_summary.csv")
print(df_summary.to_string(index=False))


# =============================================================================
# 10. 逐日匯出表（df_export）── 與你原本的 df_export 相同格式
# =============================================================================
# 組合測試集逐日資料（研究結論以測試集為主）
df_export = pd.DataFrame({
    'DATE':            dates_te,
    'ret_true':        ret_te,
    'sigma_lstm':      pred_te,        # LSTM 預測波動度
    'gk_vol_actual':   true_vol_te,    # 實際 GK vol
    'shape':           shape_te,
    'VaR_ret_95':      var95_te,
    'VaR_ret_99':      var99_te,
    'viol_95':         viol95_te,
    'viol_99':         viol99_te,
}).set_index('DATE')

df_export.to_csv(os.path.join(OUTPUT_DIR, "var_es_daily.csv"),
                 encoding='utf-8-sig')
print("✓ Saved var_es_daily.csv")


# 同樣輸出訓練集逐日
df_export_tr = pd.DataFrame({
    'DATE':            dates_tr,
    'ret_true':        ret_tr,
    'sigma_lstm':      pred_tr,
    'gk_vol_actual':   true_vol_tr,
    'shape':           shape_tr,
    'VaR_ret_95':      var95_tr,
    'VaR_ret_99':      var99_tr,
    'viol_95':         viol95_tr,
    'viol_99':         viol99_tr,
}).set_index('DATE')

df_export_tr.to_csv(os.path.join(OUTPUT_DIR, "var_es_daily_train.csv"),
                    encoding='utf-8-sig')
print("✓ Saved var_es_daily_train.csv")


# =============================================================================
# 11. 圖表輸出
# =============================================================================

# ── Fig 1: 2×2 波動預測 + VaR（訓練 & 測試）─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle('LSTM-GK: Volatility Forecast & VaR(95%) — Train vs Test',
             fontsize=14, fontweight='bold')

for ax, dates, pred, true, ret, var, viol, label, rmse, mae, corr in [
    (axes[0, 0], dates_tr, pred_tr, true_vol_tr, ret_tr, var95_tr, viol95_tr,
     'Vol Forecast — Train (2006-2021)', metrics_tr['RMSE'], metrics_tr['MAE'], metrics_tr['Corr']),
    (axes[0, 1], dates_te, pred_te, true_vol_te, ret_te, var95_te, viol95_te,
     'Vol Forecast — Test (2022-2025)',  metrics_te['RMSE'], metrics_te['MAE'], metrics_te['Corr']),
]:
    ax.plot(dates, true, color='#333', lw=0.7, alpha=0.9, label='Actual gk_vol_daily')
    ax.plot(dates, pred, color='#1565C0', lw=0.7, alpha=0.85, label='LSTM Forecast')
    ax.set_title(f'{label}\nRMSE={rmse:.5f}  MAE={mae:.5f}  Corr={corr:.4f}', fontsize=10)
    ax.set_ylabel('Volatility σ', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))

for ax, dates, ret, var, viol, label in [
    (axes[1, 0], dates_tr, ret_tr, var95_tr, viol95_tr,
     f"VaR(95%) — Train  |  Violations={viol95_tr.sum()} ({viol95_tr.mean()*100:.1f}%)"),
    (axes[1, 1], dates_te, ret_te, var95_te, viol95_te,
     f"VaR(95%) — Test   |  Violations={viol95_te.sum()} ({viol95_te.mean()*100:.1f}%)"),
]:
    ax.plot(dates, ret, color='#555', lw=0.5, alpha=0.8, label='ES1 Return')
    ax.plot(dates, var, color='#1565C0', lw=1.0, label='LSTM VaR(95%)')
    ax.fill_between(dates, var, ret.min() * 1.1, alpha=0.06, color='#1565C0')
    vmask = viol == 1
    ax.scatter(dates[vmask], ret[vmask], s=12, color='#D32F2F', zorder=5, label='Violation')
    ax.axhline(0, color='k', lw=0.4, ls='--', alpha=0.4)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Log Return', fontsize=9)
    ax.legend(loc='lower left', fontsize=8); ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_vol_and_var.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved fig1_vol_and_var.png")


# ── Fig 2: RMSE / MAE 比較柱狀圖───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('Volatility Forecast Error: Train vs Test', fontsize=13, fontweight='bold')
for ax, name, vals in [
    (axes[0], 'RMSE', [metrics_tr['RMSE'], metrics_te['RMSE']]),
    (axes[1], 'MAE',  [metrics_tr['MAE'],  metrics_te['MAE']]),
]:
    bars = ax.bar(['Train\n(2006-2021)', 'Test\n(2022-2025)'],
                  vals, color=['#1565C0', '#E53935'], width=0.4, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + max(vals) * 0.01,
                f'{v:.6f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(f'{name} Comparison', fontsize=11)
    ax.set_ylabel(f'{name} (volatility scale)', fontsize=9)
    ax.set_ylim(0, max(vals) * 1.22)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_rmse_mae.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved fig2_rmse_mae.png")


# ── Fig 3: 逐年違規率（訓練 & 測試）────────────────────────────────────────────
ann_tr = (pd.DataFrame({'year': dates_tr.year, 'viol': viol95_tr})
          .groupby('year').agg(vr=('viol', 'mean')).reset_index())
ann_te = (pd.DataFrame({'year': dates_te.year, 'viol': viol95_te})
          .groupby('year').agg(vr=('viol', 'mean')).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Annual VaR(95%) Violation Rate — Train vs Test', fontsize=13, fontweight='bold')

for ax, ann, title in [(axes[0], ann_tr, 'Train Set (2006-2021)'),
                        (axes[1], ann_te, 'Test Set (2022-2025)')]:
    cols = ['#D32F2F' if v > 0.05 else '#2E7D32' for v in ann['vr']]
    bars = ax.bar(ann['year'].astype(str), ann['vr'], color=cols, edgecolor='white')
    ax.axhline(0.05, color='navy', ls='--', lw=1.5, label='5% target')
    for bar, v in zip(bars, ann['vr']):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.002,
                f'{v*100:.1f}%', ha='center', va='bottom', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Violation Rate', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_annual_violation.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved fig3_annual_violation.png")


# ── Fig 4: 60 日滾動違規率（測試集）─────────────────────────────────────────────
df_roll = pd.DataFrame({'viol': viol95_te}, index=dates_te)
roll60  = df_roll['viol'].rolling(60).mean()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(dates_te, roll60, color='#1565C0', lw=1.5, label='LSTM-GK (60d rolling)')
ax.axhline(0.05, color='red', ls='--', lw=1.2, label='5% target')
ax.set_ylabel('Rolling Violation Rate')
ax.legend(fontsize=9); ax.grid(alpha=0.25)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.set_title('Rolling 60-Day Violation Rate — Test Set (2022-2025)', fontsize=12)
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_rolling_violation.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved fig4_rolling_violation.png")


# ── Fig 5: 訓練損失曲線 ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history['loss'],     color='#1565C0', lw=1.5, label='Train Loss')
ax.plot(history.history['val_loss'], color='#E53935', lw=1.5, ls='--', label='Val Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Huber Loss (scaled)')
ax.set_title('LSTM Training Loss Curve (EarlyStopping + ReduceLROnPlateau)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig5_loss_curve.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ Saved fig5_loss_curve.png")


# ── 原版 UC coverage 圖（Wilson CI）────────────────────────────────────────────
def wilson_ci(x: int, n: int, z: float = 1.96):
    if n <= 0:
        return (np.nan, np.nan)
    p = x / n
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half   = (z * np.sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)


def plot_uc_coverage(viol95, viol99, dates, label: str, out_path: str):
    specs = [("95%", ALPHA_95, viol95), ("99%", ALPHA_99, viol99)]
    rows = []
    for lvl, alpha, viol in specs:
        n = len(viol); x = int(viol.sum())
        p_hat = x / n if n > 0 else np.nan
        ci_lo, ci_hi = wilson_ci(x, n)
        rows.append((lvl, alpha, n, x, p_hat, ci_lo, ci_hi))
    res = pd.DataFrame(rows, columns=['level','alpha','n','x','viol_rate','ci_low','ci_high'])

    fig, ax = plt.subplots(figsize=(7, 4))
    xs = np.arange(len(res))
    y    = res['viol_rate'].values
    yerr = np.vstack([y - res['ci_low'].values, res['ci_high'].values - y])
    ax.errorbar(xs, y, yerr=yerr, fmt='o', capsize=4, label='Observed rate (95% CI)')
    ax.plot(xs, res['alpha'].values, linestyle='--', marker='s', label='Theoretical alpha')
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{lv}\n(n={n}, x={x})"
                        for lv, n, x in zip(res['level'], res['n'], res['x'])])
    ax.set_ylabel('Violation Rate')
    ax.set_title(f'UC Coverage — {label}')
    ax.grid(True, alpha=0.3); ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    return res


uc_tr = plot_uc_coverage(
    viol95_tr, viol99_tr, dates_tr, "Train 2006-2021",
    os.path.join(OUTPUT_DIR, "fig_uc_coverage_train.png"))
uc_te = plot_uc_coverage(
    viol95_te, viol99_te, dates_te, "Test 2022-2025",
    os.path.join(OUTPUT_DIR, "fig_uc_coverage_test.png"))
print("✓ Saved UC coverage figures")


# ── Hit timeline（沿用你原本的版本）────────────────────────────────────────────
def plot_hit_timeline(viol95, viol99, dates, label: str, out_path: str):
    fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
    hit95 = dates[viol95 == 1]
    hit99 = dates[viol99 == 1]
    axes[0].eventplot(hit95, lineoffsets=1, linelengths=0.8)
    axes[0].set_yticks([1]); axes[0].set_yticklabels(["VaR 95% hit"])
    axes[0].set_title(f"Hit Timeline — {label}")
    axes[0].grid(True, axis='x', alpha=0.3)
    axes[1].eventplot(hit99, lineoffsets=1, linelengths=0.8)
    axes[1].set_yticks([1]); axes[1].set_yticklabels(["VaR 99% hit"])
    axes[1].grid(True, axis='x', alpha=0.3)
    axes[1].set_xlabel("Time")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()


plot_hit_timeline(
    viol95_te, viol99_te, dates_te.values, "Test 2022-2025",
    os.path.join(OUTPUT_DIR, "hit_timeline_test.png"))
print("✓ Saved hit_timeline_test.png")


# =============================================================================
# 12. 存模型
# =============================================================================
model_path = os.path.join(OUTPUT_DIR, "lstm_gk_model.keras")
model.save(model_path)
print(f"✓ Model saved: {model_path}")

print("\n========== 全部完成 ==========")
print(f"所有輸出位於：{OUTPUT_DIR}/")
print("主要檔案：")
print("  vol_forecast_metrics.csv  — RMSE/MAE/Corr")
print("  backtest_summary.csv      — Kupiec + CC (train & test, 95% & 99%)")
print("  var_es_daily.csv          — 測試集逐日 VaR + 違規")
print("  var_es_daily_train.csv    — 訓練集逐日 VaR + 違規")
print("  fig1_vol_and_var.png      — 2×2 波動預測 + VaR 總覽")
print("  fig2_rmse_mae.png         — RMSE/MAE 比較")
print("  fig3_annual_violation.png — 逐年違規率")
print("  fig4_rolling_violation.png — 滾動違規率")
print("  fig5_loss_curve.png       — 訓練曲線")